# Chapitre 9 — DocuRAG

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-09-docurag/09_docurag.ipynb)

Ce laboratoire transforme les 18 extraits du chapitre en un projet complet : ingestion, retrieval hybride, génération citée, API, interface, évaluation et déploiement.

## Ressources utiles

- [OpenAI Docs — génération de texte](https://developers.openai.com/api/docs/guides/text-generation)
- [OpenAI Docs — embeddings](https://developers.openai.com/api/docs/guides/embeddings)
- [Qdrant — documentation](https://qdrant.tech/documentation/)
- [FastAPI — documentation](https://fastapi.tiangolo.com/)
- [Streamlit — documentation](https://docs.streamlit.io/)

## 0. Choisir un fournisseur

OpenAI et Hugging Face fonctionnent dans Colab. Ollama est destiné à l'exécution locale, avec le serveur démarré avant le notebook.

In [ ]:
# @title Choisir le fournisseur de modèles
PROVIDER = "openai" # @param ["openai", "huggingface", "ollama"]
PROVIDER = PROVIDER.strip().lower()
if PROVIDER not in {"openai", "huggingface", "ollama"}:
    raise ValueError("Choisissez openai, huggingface ou ollama")
print("Fournisseur choisi :", PROVIDER)


## Préparer le dépôt et les dépendances

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

provider_extra = {"openai": "openai", "huggingface": "huggingface", "ollama": ""}[PROVIDER]
extras = ["app", "pdf", "retrieval"]
if provider_extra:
    extras.append(provider_extra)
target = f".[{','.join(extras)}]"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", target], check=True)
sys.path.insert(0, str(Path("src").resolve()))
sys.path.insert(0, str(Path("chapters/chapitre-09-docurag/runnable").resolve()))
print("Environnement DocuRAG prêt :", Path.cwd())


## Configurer le fournisseur

La clé OpenAI est demandée de manière masquée et reste uniquement en mémoire.

In [ ]:
import os
from getpass import getpass

os.environ["RAG_PROVIDER"] = PROVIDER
if PROVIDER == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()
    if not os.environ["OPENAI_API_KEY"] or not os.environ["OPENAI_MODEL"]:
        raise RuntimeError("OPENAI_API_KEY et OPENAI_MODEL sont obligatoires")
elif PROVIDER == "huggingface":
    os.environ.setdefault(
        "HF_EMBEDDING_MODEL",
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    )
    os.environ.setdefault("HF_GENERATION_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
else:
    os.environ.setdefault("OLLAMA_BASE_URL", "http://localhost:11434")
    os.environ.setdefault("OLLAMA_EMBEDDING_MODEL", "embeddinggemma")
    os.environ.setdefault("OLLAMA_MODEL", "qwen3:0.6b")
    print("Ollama doit déjà être démarré et les deux modèles téléchargés.")
print("Configuration chargée pour", PROVIDER)


## 1. Architecture du projet

Repérer les couches hors ligne, en ligne et d'interface.

Fichier correspondant : [`01_arborescence.txt`](examples/01_arborescence.txt)

```text
docurag/
|-- src/
|   |-- config.py            # Configuration centralisee (unique source)
|   |-- ingestion/           # Chargement et préparation
|   |   |-- loader.py        #   chargement multi-sources
|   |   |-- chunker.py       #   decoupage
|   |   |-- indexer.py       #   vectorisation + ecriture dans Qdrant
|   |   |-- __init__.py      #   orchestration du pipeline
|   |-- retrieval/           # === EN LIGNE ===
|   |   |-- retriever.py     #   hybride + fusion + re-ranking
|   |-- generation/          # === EN LIGNE ===
|   |   |-- prompts.py       #   gabarits versionnes
|   |   |-- generator.py     #   appel du modele, citations, confiance
|   |-- api/
|   |   |-- schemas.py       #   contrats d'entree/sortie (Pydantic)
|   |   |-- main.py          #   application FastAPI
|   |-- interface/
|       |-- app.py           #   interface Streamlit
|-- tests/
|   |-- test_ingestion.py
|   |-- test_retrieval.py
|   |-- test_evaluation.py   # evaluation RAGAS, lancee en CI
|-- documents/               # corpus a indexer (hors depot Git)
|-- eval/
|   |-- jeu_reference.json   # le jeu du chapitre precedent, versionne
|-- docker/
|   |-- Dockerfile
|   |-- docker-compose.yml
|-- .env.example             # modele de configuration, SANS secrets
|-- .gitignore               # contient .env et documents/
|-- requirements.txt
|-- README.md

```

## 2. Configuration centralisée

Valider les paramètres avant de charger les modèles.

Fichier correspondant : [`02_config.py`](examples/02_config.py)

In [ ]:
# ruff: noqa: E402, F811
"""Configuration centralisée de DocuRAG."""

from __future__ import annotations

import os
from dataclasses import dataclass


@dataclass(frozen=True)
class Settings:
    provider: str = "openai"
    generation_model: str | None = None
    chunk_size: int = 80
    chunk_overlap: int = 15
    initial_k: int = 10
    top_k: int = 3
    score_threshold: float = 0.15

    def __post_init__(self) -> None:
        if self.provider not in {"openai", "huggingface", "ollama"}:
            raise ValueError("RAG_PROVIDER invalide")
        if not 0 <= self.chunk_overlap < self.chunk_size:
            raise ValueError("DOCURAG_CHUNK_OVERLAP doit être inférieur à la taille")
        if self.initial_k < self.top_k:
            raise ValueError("DOCURAG_INITIAL_K doit être supérieur ou égal à DOCURAG_TOP_K")

    @classmethod
    def from_env(cls) -> Settings:
        provider = os.getenv("RAG_PROVIDER", "openai").lower()
        model_variables = {
            "openai": "OPENAI_MODEL",
            "huggingface": "HF_GENERATION_MODEL",
            "ollama": "OLLAMA_MODEL",
        }
        return cls(
            provider=provider,
            generation_model=os.getenv(model_variables.get(provider, "")) or None,
            chunk_size=int(os.getenv("DOCURAG_CHUNK_SIZE", "80")),
            chunk_overlap=int(os.getenv("DOCURAG_CHUNK_OVERLAP", "15")),
            initial_k=int(os.getenv("DOCURAG_INITIAL_K", "10")),
            top_k=int(os.getenv("DOCURAG_TOP_K", "3")),
            score_threshold=float(os.getenv("DOCURAG_SCORE_THRESHOLD", "0.15")),
        )


if __name__ == "__main__":
    settings = Settings.from_env()
    print(settings)


## 3. Variables d'environnement

Séparer la configuration des secrets et du code.

Fichier correspondant : [`03_env_example.txt`](examples/03_env_example.txt)

```text
# Configuration DocuRAG
# Copier ces variables dans un fichier .env local non versionné.

# Choisir : openai, huggingface ou ollama
RAG_PROVIDER=openai

# OpenAI (requis uniquement si RAG_PROVIDER=openai)
OPENAI_API_KEY=
OPENAI_MODEL=
OPENAI_EMBEDDING_MODEL=text-embedding-3-small

# Hugging Face local/Colab (requis uniquement si RAG_PROVIDER=huggingface)
HF_EMBEDDING_MODEL=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
HF_GENERATION_MODEL=Qwen/Qwen2.5-0.5B-Instruct
HF_MAX_NEW_TOKENS=256

# Ollama local (requis uniquement si RAG_PROVIDER=ollama)
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_EMBEDDING_MODEL=embeddinggemma
OLLAMA_MODEL=qwen3:0.6b

DOCURAG_CHUNK_SIZE=80
DOCURAG_CHUNK_OVERLAP=15
DOCURAG_INITIAL_K=10
DOCURAG_TOP_K=3
DOCURAG_SCORE_THRESHOLD=0.15
DOCURAG_HYBRID=true
DOCURAG_API_URL=http://localhost:8000

```

## 4. Dépendances

Installer seulement les extras utiles au fournisseur choisi.

Fichier correspondant : [`04_requirements.txt`](examples/04_requirements.txt)

```text
# Profil DocuRAG avec OpenAI, API, interface, PDF et retrieval hybride
-e .[openai,app,pdf,retrieval]

# Pour Hugging Face, remplacer "openai" par "huggingface".
# Pour Ollama, retirer "openai" puis démarrer Ollama séparément.
# Le profil complet reste disponible avec : pip install -e ".[all]"

```

## 5. Chargement tolérant

Enrichir chaque document avec sa source, son département et son empreinte.

Fichier correspondant : [`05_loader.py`](examples/05_loader.py)

In [ ]:
# ruff: noqa: E402, F811
"""Chargement tolérant avec métadonnées et empreinte de contenu."""

from __future__ import annotations

import hashlib
from pathlib import Path

from rag_en_pratique.core import Document

KNOWN_DEPARTMENTS = {"rh", "finance", "juridique", "it", "produit", "commercial"}


def empreinte(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(65_536), b""):
            digest.update(block)
    return digest.hexdigest()


def departement(path: Path) -> str:
    return next((part for part in path.parts if part.lower() in KNOWN_DEPARTMENTS), "general")


def charger_dossier(directory: Path) -> list[Document]:
    if not directory.is_dir():
        raise FileNotFoundError(f"Dossier introuvable : {directory}")
    documents: list[Document] = []
    for path in sorted(directory.rglob("*")):
        if (
            path.is_file()
            and path.suffix.lower() in {".md", ".txt"}
            and path.name.lower() != "readme.md"
        ):
            try:
                text = path.read_text(encoding="utf-8")
            except (OSError, UnicodeError) as error:
                print(f"Ignoré : {path.name} ({error})")
                continue
            documents.append(
                Document(
                    text,
                    {
                        "source": path.name,
                        "source_path": str(path),
                        "department": departement(path),
                        "fingerprint": empreinte(path),
                    },
                )
            )
    return documents


if __name__ == "__main__":
    print(f"{len(charger_dossier(Path('data/sample')))} document(s) chargé(s)")


## 6. Découpage traçable

Préserver les métadonnées et numéroter chaque chunk.

Fichier correspondant : [`06_chunker.py`](examples/06_chunker.py)

In [ ]:
# ruff: noqa: E402, F811
"""Découpage des documents avant l'indexation OpenAI."""

from __future__ import annotations

from collections.abc import Iterable

from rag_en_pratique.core import Document, split_documents


def decouper(
    documents: Iterable[Document],
    *,
    chunk_size: int = 80,
    overlap: int = 15,
) -> list[Document]:
    chunks = split_documents(documents, chunk_size=chunk_size, overlap=overlap)
    return [
        Document(
            chunk.text,
            {**chunk.metadata, "chunk_id": index, "chunk_size": len(chunk.text)},
        )
        for index, chunk in enumerate(chunks)
    ]


if __name__ == "__main__":
    sample = Document(" ".join(f"mot-{index}" for index in range(120)), {"source": "demo.txt"})
    for chunk in decouper([sample], chunk_size=40, overlap=8):
        print(chunk.metadata, chunk.text[:50])


## 7. Indexation vectorielle

Injecter l'embedder OpenAI, Hugging Face ou Ollama.

Fichier correspondant : [`07_indexer.py`](examples/07_indexer.py)

In [ ]:
# ruff: noqa: E402, F811
"""Indexation vectorielle de DocuRAG avec le fournisseur configuré."""

from __future__ import annotations

from collections.abc import Sequence
from pathlib import Path

from rag_en_pratique.core import Document, InMemoryVectorStore
from rag_en_pratique.providers import create_embedder


class Indexeur:
    """Index local pédagogique ; remplaçable par Qdrant en production."""

    def __init__(self, provider: str | None = None, *, client=None) -> None:
        self.store = InMemoryVectorStore(create_embedder(provider, client=client))

    def indexer(self, chunks: Sequence[Document]) -> int:
        self.store.add(chunks)
        return len(chunks)


def main() -> None:
    chunks = [
        Document(path.read_text(encoding="utf-8"), {"source": path.name})
        for path in sorted(Path("data/sample").glob("*.md"))
        if path.name.lower() != "readme.md"
    ]
    print(f"{Indexeur().indexer(chunks)} document(s) indexé(s)")


if __name__ == "__main__":
    main()


## 8. Pipeline d'ingestion

Assembler chargement, découpage et indexation.

Fichier correspondant : [`08_pipeline_ingestion.py`](examples/08_pipeline_ingestion.py)

In [ ]:
# ruff: noqa: E402, F811
"""Pipeline d'ingestion DocuRAG utilisant le fournisseur configuré."""

from __future__ import annotations

from collections.abc import Iterable
from pathlib import Path

from rag_en_pratique.core import Document, InMemoryVectorStore, split_documents
from rag_en_pratique.providers import create_embedder


def ingerer(
    documents: Iterable[Document],
    *,
    provider: str | None = None,
    client=None,
) -> InMemoryVectorStore:
    chunks = split_documents(documents, chunk_size=80, overlap=15)
    store = InMemoryVectorStore(create_embedder(provider, client=client))
    store.add(chunks)
    return store


def main() -> None:
    documents = [
        Document(path.read_text(encoding="utf-8"), {"source": path.name})
        for path in sorted(Path("data/sample").glob("*.md"))
        if path.name.lower() != "readme.md"
    ]
    store = ingerer(documents)
    print(f"{len(store.documents)} chunk(s) indexé(s)")


if __name__ == "__main__":
    main()


## 9. Retriever hybride

Fusionner recherche dense et lexicale par les rangs.

Fichier correspondant : [`09_retriever.py`](examples/09_retriever.py)

In [ ]:
# ruff: noqa: E402, F811
"""Retrieval hybride : classement dense, lexical puis fusion RRF."""

from __future__ import annotations

from collections.abc import Sequence

from rag_en_pratique.core import Document, InMemoryVectorStore, SearchResult
from rag_en_pratique.providers import create_embedder


def fusion_rrf(
    rankings: Sequence[Sequence[SearchResult]],
    *,
    k: int = 60,
) -> list[SearchResult]:
    scores: dict[int, float] = {}
    documents: dict[int, Document] = {}
    for ranking in rankings:
        for rank, result in enumerate(ranking, start=1):
            key = id(result.document)
            documents[key] = result.document
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
    return sorted(
        (SearchResult(documents[key], score) for key, score in scores.items()),
        key=lambda result: result.score,
        reverse=True,
    )


def retrieve(
    store: InMemoryVectorStore,
    question: str,
    *,
    top_k: int = 5,
) -> list[SearchResult]:
    dense = store.search(question, top_k=top_k * 2)
    query_words = set(question.lower().split())
    lexical = sorted(
        (
            SearchResult(document, len(query_words & set(document.text.lower().split())))
            for document in store.documents
        ),
        key=lambda result: result.score,
        reverse=True,
    )
    return fusion_rrf([dense, lexical])[:top_k]


def main() -> None:
    store = InMemoryVectorStore(create_embedder())
    store.add(
        [
            Document("Les retours sont acceptés sous 30 jours.", {"source": "retours.md"}),
            Document("La livraison prend trois à cinq jours.", {"source": "livraison.md"}),
        ]
    )
    for result in retrieve(store, "Quel est le délai de retour ?", top_k=2):
        print(round(result.score, 4), result.document.metadata["source"])


if __name__ == "__main__":
    main()


## 10. Prompts versionnés

Rendre les règles d'ancrage et d'abstention testables.

Fichier correspondant : [`10_prompts.py`](examples/10_prompts.py)

In [ ]:
# ruff: noqa: E402, F811
"""Gabarits versionnés de DocuRAG."""

VERSION_PROMPT = "1.2"

SYSTEME = """Tu es {app_name}, un assistant documentaire rigoureux.

1. Tu réponds EXCLUSIVEMENT à partir des extraits fournis.
2. Si l'information ne s'y trouve pas, écris exactement :
   "Cette information ne figure pas dans les documents consultés."
3. Fais suivre chaque affirmation de sa source : [doc_N].
4. Si les extraits ne couvrent qu'une partie de la question, précise ce qui manque.
5. Réponds en français, de manière concise et professionnelle.

Toute affirmation doit pouvoir être reliée à un extrait."""

UTILISATEUR = """EXTRAITS :
{contexte}

QUESTION : {question}

RÉPONSE :"""


def construire_prompt(app_name: str, contexte: str, question: str) -> tuple[str, str]:
    return SYSTEME.format(app_name=app_name), UTILISATEUR.format(
        contexte=contexte,
        question=question,
    )


if __name__ == "__main__":
    systeme, utilisateur = construire_prompt(
        "DocuRAG",
        "[doc_1] retours.md\nLes retours sont acceptés sous 30 jours.",
        "Quel est le délai de retour ?",
    )
    print(systeme, utilisateur, sep="\n\n")


## 11. Réponse citée

Produire réponse, sources, confiance et version du prompt.

Fichier correspondant : [`11_generator.py`](examples/11_generator.py)

In [ ]:
# ruff: noqa: E402, F811
"""Génération citée avec abstention et niveau de confiance."""

from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass, field

from rag_en_pratique.core import Document, SearchResult
from rag_en_pratique.providers import create_generator

ABSTENTION = "Cette information ne figure pas dans les documents consultés."
PROMPT_VERSION = "1.2"


@dataclass
class Reponse:
    texte: str
    sources: list[dict[str, object]] = field(default_factory=list)
    confiance: str = "Bas"
    version_prompt: str = PROMPT_VERSION


class Generateur:
    def __init__(
        self,
        provider: str | None = None,
        model: str | None = None,
        *,
        client=None,
    ) -> None:
        self.generator = create_generator(provider, model=model, client=client)

    def repondre(self, question: str, passages: Sequence[SearchResult]) -> Reponse:
        if not passages:
            return Reponse(ABSTENTION)
        meilleur = max(passage.score for passage in passages)
        confiance = "Haut" if meilleur >= 0.7 else "Moyen" if meilleur >= 0.4 else "Bas"
        sources = [
            {
                "document": passage.document.metadata.get("source", "document"),
                "pertinence": round(passage.score, 3),
            }
            for passage in passages
        ]
        return Reponse(
            self.generator.generate(question, passages),
            sources=sources,
            confiance=confiance,
        )


def main() -> None:
    passages = [
        SearchResult(
            Document("La livraison prend trois à cinq jours.", {"source": "livraison.md"}),
            1.0,
        )
    ]
    response = Generateur().repondre("Quel est le délai de livraison ?", passages)
    print(response.texte)
    print(response.sources, response.confiance)


if __name__ == "__main__":
    main()


## 12. Contrats Pydantic

Valider les entrées et stabiliser les sorties de l'API.

Fichier correspondant : [`12_schemas.py`](examples/12_schemas.py)

In [ ]:
# ruff: noqa: E402, F811
"""Contrats Pydantic de l'API DocuRAG."""

from __future__ import annotations

from pydantic import BaseModel, Field


class QuestionRequest(BaseModel):
    question: str = Field(min_length=3, max_length=2_000)
    department: str | None = None


class SourceResponse(BaseModel):
    document: str
    page: int = 0
    department: str = "general"
    relevance: float


class AnswerResponse(BaseModel):
    answer: str
    sources: list[SourceResponse] = Field(default_factory=list)
    confidence: str
    passage_count: int
    prompt_version: str
    duration_ms: int


class IngestionRequest(BaseModel):
    rebuild: bool = False
    path: str = "data/sample"


if __name__ == "__main__":
    request = QuestionRequest(question="Quel est le délai de livraison ?")
    print(request.model_dump())


## 13. API FastAPI

Exposer santé, ingestion et interrogation.

Fichier correspondant : [`13_api_main.py`](examples/13_api_main.py)

In [ ]:
# ruff: noqa: E402, F811
"""API FastAPI exposant ingestion, santé et interrogation DocuRAG."""

from __future__ import annotations

import sys
import time
from pathlib import Path

from fastapi import FastAPI
from pydantic import BaseModel, Field

RUNNABLE = Path("chapters/chapitre-09-docurag/runnable").resolve()
sys.path.insert(0, str(RUNNABLE))

from docurag import DocuRAG


class QueryRequest(BaseModel):
    question: str = Field(min_length=3)
    department: str | None = None


class IngestionRequest(BaseModel):
    rebuild: bool = False


def build_pipeline() -> DocuRAG:
    docurag = DocuRAG()
    docurag.ingest(Path("data/sample"), rebuild=True)
    return docurag


app = FastAPI(title="DocuRAG", version="1.0.0")
pipeline: DocuRAG | None = None


@app.on_event("startup")
def startup() -> None:
    global pipeline
    pipeline = build_pipeline()


@app.get("/health")
def health() -> dict[str, object]:
    return {
        "status": "ok" if pipeline else "initializing",
        "indexed_chunks": len(pipeline.store.documents) if pipeline else 0,
    }


@app.post("/ingest")
def ingest(request: IngestionRequest) -> dict[str, int]:
    if pipeline is None:
        raise RuntimeError("Le pipeline n'est pas initialisé")
    repository = Path.cwd()
    count = pipeline.ingest(repository / "data" / "sample", rebuild=request.rebuild)
    return {"indexed_chunks": count}


@app.post("/query")
def query(request: QueryRequest) -> dict[str, object]:
    if pipeline is None:
        raise RuntimeError("Le pipeline n'est pas initialisé")
    started = time.perf_counter()
    filters = {"department": request.department} if request.department else None
    result = pipeline.ask(request.question, filters=filters)
    result["duration_ms"] = round((time.perf_counter() - started) * 1_000)
    return result


## 14. Interface Streamlit

Afficher réponse, confiance, latence et sources.

Fichier correspondant : [`14_interface.py`](examples/14_interface.py)

In [ ]:
# ruff: noqa: E402, F811
"""Interface Streamlit connectée à l'API DocuRAG."""

from __future__ import annotations

import os

import requests
import streamlit as st

API_URL = os.getenv("DOCURAG_API_URL", "http://localhost:8000")

st.set_page_config(page_title="DocuRAG", page_icon="📚")
st.title("DocuRAG — assistant documentaire")

department = st.sidebar.selectbox(
    "Département",
    ["Tous", "RH", "finance", "juridique", "IT", "produit"],
)

question = st.chat_input("Posez votre question sur les documents…")
if question:
    with st.chat_message("user"):
        st.write(question)
    with st.chat_message("assistant"):
        with st.spinner("Recherche et génération…"):
            payload = {"question": question}
            if department != "Tous":
                payload["department"] = department
            response = requests.post(
                f"{API_URL}/query",
                json=payload,
                timeout=120,
            )
            response.raise_for_status()
            payload = response.json()
        st.write(payload["answer"])
        st.caption(
            f"Confiance : {payload['confidence']} — "
            f"{payload['passage_count']} passage(s) — {payload['duration_ms']} ms"
        )
        with st.expander("Sources"):
            for source in payload["sources"]:
                st.caption(
                    f"{source['document']} — pertinence {source['relevance']}"
                )


## 15. Garde-barrière d'évaluation

Comparer les métriques à une ligne de base mesurée.

Fichier correspondant : [`15_test_evaluation.py`](examples/15_test_evaluation.py)

In [ ]:
# ruff: noqa: E402, F811
"""Évaluation d'une réponse RAG par le fournisseur configuré."""

from __future__ import annotations

from rag_en_pratique.core import Document, SearchResult
from rag_en_pratique.providers import create_generator


def evaluate_grounding(
    question: str,
    context: str,
    answer: str,
    *,
    provider: str | None = None,
    client=None,
) -> str:
    judge_question = (
        "Évalue si la réponse candidate est entièrement fondée sur le contexte. "
        "Réponds par VALIDE ou INVALIDE, puis justifie brièvement.\n\n"
        f"Question initiale : {question}\nRéponse candidate : {answer}"
    )
    passage = SearchResult(Document(context, {"source": "contexte_evaluation"}), 1.0)
    return create_generator(provider, client=client).generate(judge_question, [passage])


def verifier_seuils(scores: dict[str, float], thresholds: dict[str, float]) -> None:
    failures = [
        f"{name}={scores.get(name, 0):.3f} < {threshold:.3f}"
        for name, threshold in thresholds.items()
        if scores.get(name, 0) < threshold
    ]
    if failures:
        raise AssertionError("Régression détectée :\n" + "\n".join(failures))


if __name__ == "__main__":
    verdict = evaluate_grounding(
        "Quel est le délai de retour ?",
        "Les retours sont acceptés sous 30 jours.",
        "Le délai de retour est de 30 jours.",
    )
    print(verdict)


## 16. Image Docker

Construire une image reproductible de l'API.

Fichier correspondant : [`16_dockerfile.txt`](examples/16_dockerfile.txt)

```text
# docker/Dockerfile
FROM python:3.11-slim

LABEL description="DocuRAG - assistant documentaire RAG"

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1

WORKDIR /app

# Dépendances système requises par les loaders de documents
RUN apt-get update && apt-get install -y --no-install-recommends \
        libmagic1 poppler-utils curl \
    && rm -rf /var/lib/apt/lists/*

# Le projet compagnon utilise pyproject.toml et ses extras.
COPY pyproject.toml README.md ./
COPY src/ ./src/
COPY chapters/chapitre-09-docurag/runnable/ ./chapters/chapitre-09-docurag/runnable/
COPY chapters/chapitre-09-docurag/examples/13_api_main.py ./api_main.py
COPY data/sample/ ./data/sample/
RUN pip install --no-cache-dir -e ".[openai,app,pdf,retrieval]"

# start-period genereux : le premier demarrage telecharge le
# modele d'embedding, ce qui prend du temps.
HEALTHCHECK --interval=30s --timeout=10s --start-period=180s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000

CMD ["uvicorn", "api_main:app", \
     "--host", "0.0.0.0", "--port", "8000"]

```

## 17. Pile Docker Compose

Orchestrer Qdrant, API et interface.

Fichier correspondant : [`17_compose.yml`](examples/17_compose.yml)

```yaml
# docker/docker-compose.yml
services:

  qdrant:
    image: qdrant/qdrant:latest
    container_name: docurag-qdrant
    restart: unless-stopped
    ports:
      - "6333:6333"
    volumes:
      # SANS ce volume, l'index disparait a chaque redemarrage
      # et il faut tout reindexer. C'est la ligne qu'on oublie.
      - qdrant_data:/qdrant/storage
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:6333/readyz"]
      interval: 10s
      timeout: 5s
      retries: 5

  api:
    build:
      context: ..
      dockerfile: docker/Dockerfile
    container_name: docurag-api
    restart: unless-stopped
    ports:
      - "8000:8000"
    env_file:
      - ../.env
    environment:
      # Dans le reseau Docker, "localhost" designe le conteneur
      # lui-meme. Il faut le NOM DU SERVICE. C'est la cause
      # numero un des echecs de premier demarrage.
      QDRANT_URL: http://qdrant:6333
      RAG_PROVIDER: ${RAG_PROVIDER:-openai}
    volumes:
      - ../documents:/app/documents:ro    # lecture seule
    depends_on:
      qdrant:
        condition: service_healthy        # attend que Qdrant reponde

  interface:
    build:
      context: ..
      dockerfile: docker/Dockerfile
    container_name: docurag-interface
    restart: unless-stopped
    ports:
      - "8501:8501"
    environment:
      DOCURAG_API_URL: http://api:8000
    command: >
      streamlit run src/interface/app.py
      --server.port 8501 --server.address 0.0.0.0
    depends_on:
      - api

volumes:
  qdrant_data:

```

## 18. Séquence de démarrage

Déployer puis déclencher l'ingestion complète ou incrémentale.

Fichier correspondant : [`18_demarrage.sh`](examples/18_demarrage.sh)

```bash
# 1. Configuration
cp .env.example .env
# Editer .env : renseigner la cle d'API du modele

# 2. Deposer les documents a indexer
cp -r /chemin/vers/vos/documents/* ./documents/

# 3. Demarrer la pile
cd docker && docker compose up -d --build

# 4. Attendre que tout soit en bonne sante
docker compose ps
# docurag-qdrant     running (healthy)
# docurag-api        running (healthy)   <- peut prendre 2-3 min
# docurag-interface  running

# 5. Premiere ingestion (complete)
curl -X POST http://localhost:8000/ingest \
     -H "Content-Type: application/json" \
     -d '{"tout_reconstruire": true}'

# Suivre l'avancement
docker compose logs -f api

# 6. Verifier
curl -X POST http://localhost:8000/query \
     -H "Content-Type: application/json" \
     -d '{"question": "Quelle est la politique de conges ?"}'

# 7. Interface : http://localhost:8501
#    Documentation de l'API : http://localhost:8000/docs

# --- Mises a jour ulterieures : incrementales ---
curl -X POST http://localhost:8000/ingest \
     -H "Content-Type: application/json" \
     -d '{"tout_reconstruire": false}'

```

## Exécuter le projet intégré

In [ ]:
from pathlib import Path

from docurag import DocuRAG
from docurag.config import Settings

app = DocuRAG(Settings.from_env())
chunk_count = app.ingest(Path("data/sample"), rebuild=True)
print(f"{chunk_count} chunk(s) indexé(s)")

result = app.ask("Quel est le délai de livraison standard ?")
print(result["answer"])
print("Confiance :", result["confidence"])
print("Sources :", result["sources"])


## Bilan

DocuRAG sépare les composants pour qu'ils puissent évoluer indépendamment. Le runner en mémoire sert à apprendre et à tester ; les fichiers Docker et Compose montrent la cible de déploiement avec une base vectorielle persistante.